> **Superseded.** FB15k-237 schema runs take ~5.07h each (measured), so 8 runs (4 variants x 2 new seeds) don't fit a shared session plan well. Use the 8 split notebooks (`05_FB15k237_*_seed*.ipynb`) in this folder instead, one run each -- see `00_PARALLEL_RUN_PLAN.md` for which to run given your current remaining GPU quota per account.

# BRGC Revision — Notebook 05: Schema ablation, multi-seed (GPU)

**Purpose (Reviewer 3, Reviewer 2).** The schema ablation (S0 context-only, S1 +schema tokens, S2 +schema prior, S3 full) was reported for FB15k-237 at **seed 42 only**. This adds **seeds 13 and 2025** so it is no longer single-seed; merge with the existing seed-42 row.

**Attach:** `brgc-frozen-source`, `brgc-kg-splits`. **GPU on.** FB15k-237 ~4-6h/run (4 variants x 2 seeds = 8 runs — split across sessions; 1-2 runs per 12h session).

Send me `schema_ablation_multiseed_<DATASET>.csv`.

In [ ]:
# === GPU + Hugging Face setup (run first) ===
import os, sys, subprocess, importlib.metadata as md, gc
from pathlib import Path
import torch
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle T4 GPU before running.")
PINNED={"transformers":"4.46.3","huggingface_hub":"0.26.5","tokenizers":"0.20.3","accelerate":"0.34.2","safetensors":"0.4.5"}
cur={}
for p in PINNED:
    try: cur[p]=md.version(p)
    except md.PackageNotFoundError: cur[p]=None
if any(cur[n]!=v for n,v in PINNED.items()):
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",*[f"{n}=={v}" for n,v in PINNED.items()]])
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
HF_CACHE_ROOT=Path("/kaggle/working/hf_cache"); HF_CACHE_ROOT.mkdir(parents=True,exist_ok=True)
HF_LOCAL_MODEL_DIR=Path("/kaggle/working/hf_models/distilbert-base-uncased"); HF_LOCAL_MODEL_DIR.parent.mkdir(parents=True,exist_ok=True)
os.environ["HF_HOME"]=str(HF_CACHE_ROOT); os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
snapshot_download(repo_id="distilbert-base-uncased", local_dir=str(HF_LOCAL_MODEL_DIR), cache_dir=str(HF_CACHE_ROOT), max_workers=1)
AutoConfig.from_pretrained(str(HF_LOCAL_MODEL_DIR), local_files_only=True)
AutoTokenizer.from_pretrained(str(HF_LOCAL_MODEL_DIR), use_fast=True, local_files_only=True)
_m=AutoModelForSequenceClassification.from_pretrained(str(HF_LOCAL_MODEL_DIR), local_files_only=True); del _m; gc.collect()
print("DistilBERT ready.")

In [ ]:
# === Stage frozen source + data (attach brgc-frozen-source and brgc-kg-splits) ===
import shutil
from pathlib import Path
INPUT_ROOT=Path("/kaggle/input"); PROJECT_DIR=Path("/kaggle/working/project"); (PROJECT_DIR/"data").mkdir(parents=True,exist_ok=True)
SRC=["config.py","data_loader.py","model.py","train.py","utils.py"]; src_dir=None
for p in INPUT_ROOT.rglob("config.py"):
    if all((p.parent/f).exists() for f in SRC): src_dir=p.parent; break
assert src_dir, "attach brgc-frozen-source"
for f in SRC: shutil.copy2(src_dir/f, PROJECT_DIR/f)
found={}
for name in ["CoDEx-S","CoDEx-M","FB15k-237"]:
    for c in INPUT_ROOT.rglob(name):
        if c.is_dir() and all((c/s).exists() for s in ["train.txt","valid.txt","test.txt"]):
            d=PROJECT_DIR/"data"/name; d.mkdir(parents=True,exist_ok=True)
            for s in ["train.txt","valid.txt","test.txt"]: shutil.copy2(c/s,d/s)
            found[name]=d; break
import subprocess,sys
assert subprocess.run([sys.executable,"-m","py_compile",*SRC],cwd=PROJECT_DIR).returncode==0
print("staged frozen_source + data:", sorted(found))

In [ ]:
# === Schema ablation: config, seeds, variants ===
# Reviewers note the schema ablation was single-seed (FB15k-237 seed 42). This adds seeds 13 and 2025.
DATASET_NAME = "FB15k-237"
SEEDS = [13, 2025]        # seed 42 is already in the paper; add these two then merge
# variant -> (use_schema_tokens, use_schema_prior)
VARIANTS = {"S0_context_only":(0,0), "S1_schema_tokens":(1,0), "S2_schema_prior":(0,1), "S3_full_schema":(1,1)}
assert DATASET_NAME in found
print(DATASET_NAME, SEEDS, list(VARIANTS))

In [ ]:
# === Run schema ablation (EXPERIMENT_ID=B2 + schema flags) ===
import os, sys, subprocess, json, time, shutil
from pathlib import Path
rows=[]; n=0
for vname,(tok,prior) in VARIANTS.items():
    for seed in SEEDS:
        n+=1
        out_root=PROJECT_DIR/"outputs_schema"/vname
        out_dir=out_root/DATASET_NAME/"strict_final"/"B2"/f"seed_{seed}"
        if (out_dir/"run_complete.json").exists(): print("SKIP",vname,seed)
        else:
            if out_dir.exists(): shutil.rmtree(out_dir)
            env=os.environ.copy()
            env.update({"KGC_DATASET":DATASET_NAME,"KGC_DATA_ROOT":str(PROJECT_DIR/"data"),
                "KGC_OUTPUT_ROOT":str(out_root),"KGC_EXPERIMENT_ID":"B2","KGC_SEED":str(seed),
                "KGC_DETERMINISTIC":"1","KGC_NUM_WORKERS":"0","TOKENIZERS_PARALLELISM":"false","MASTER_PORT":str(29700+n),
                "KGC_MODEL_NAME":str(HF_LOCAL_MODEL_DIR),"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","HF_HOME":str(HF_CACHE_ROOT),
                "KGC_USE_SCHEMA_TOKENS":str(tok),"KGC_USE_SCHEMA_PRIOR":str(prior)})
            t0=time.time(); lp=out_root/f"log_{vname}_seed{seed}.log"; lp.parent.mkdir(parents=True,exist_ok=True)
            with lp.open("w") as lf:
                pr=subprocess.run([sys.executable,"train.py"],cwd=PROJECT_DIR,env=env,stdout=lf,stderr=subprocess.STDOUT,text=True)
            if pr.returncode!=0: print(lp.read_text()[-5000:]); raise RuntimeError(f"failed {vname} {seed}")
            m=[json.loads(l) for l in (out_dir/"relation_metrics.jsonl").read_text().splitlines() if l.strip()][-1]
            (out_dir/"test_metrics.json").write_text(json.dumps(m,indent=2))
            (out_dir/"run_complete.json").write_text(json.dumps({"v":vname,"seed":seed,"min":(time.time()-t0)/60}))
            print(f"DONE {vname} seed={seed} FilteredMRR={m.get('FilteredMRR'):.4f}")
        m=json.loads((out_dir/"test_metrics.json").read_text())
        rows.append([DATASET_NAME,vname,seed,m.get("FilteredMRR"),m.get("FilteredHits@1"),m.get("MacroF1")])
import pandas as pd
df=pd.DataFrame(rows,columns=["Dataset","Variant","Seed","FilteredMRR","FilteredHits@1","MacroF1"])
df.to_csv(Path("/kaggle/working")/f"schema_ablation_multiseed_{DATASET_NAME}.csv",index=False)
print(df.to_string(index=False)); print("\nWrote schema_ablation_multiseed_"+DATASET_NAME+".csv  (send me this)")